# Анализ лояльности пользователей Яндекс Афиши

In [ ]:
#Импортируем библиотеки
!pip install sqlalchemy
!pip install psycopg2-binary
!pip install phik
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from phik import phik_matrix
from sqlalchemy import create_engine

     |████████████████████████████████| 677 kB 2.4 MB/s eta 0:00:01


In [ ]:
!pip install python-dotenv

In [ ]:
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

In [ ]:
engine = create_engine(connection_string)

In [ ]:

query = '''
WITH set_config_precode AS (
  SELECT set_config('synchronize_seqscans', 'off', true)
)
SELECT 
    p.user_id,
    p.device_type_canonical,
    p.order_id,
    p.created_dt_msk AS order_dt,
    p.created_ts_msk AS order_ts,
    p.currency_code,
    p.revenue,
    p.tickets_count,
    EXTRACT(DAY FROM (
        p.created_dt_msk - LAG(p.created_dt_msk) OVER (
            PARTITION BY p.user_id 
            ORDER BY p.created_dt_msk
        )
    ))::INTEGER AS days_since_prev,
    p.event_id,
    e.event_name_code AS event_name,
    e.event_type_main,
    p.service_name,
    r.region_name,
    c.city_name
FROM afisha.purchases p
LEFT JOIN afisha.events e ON p.event_id = e.event_id
LEFT JOIN afisha.city c ON e.city_id = c.city_id
LEFT JOIN afisha.regions r ON c.region_id = r.region_id
WHERE 
    p.device_type_canonical IN ('mobile', 'desktop')
    AND e.event_type_main != 'фильм'
ORDER BY p.user_id ASC;
'''

In [ ]:
df = pd.read_sql_query(query, con=engine)

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [ ]:
#Информация о датасете
df.info()

In [ ]:
#Первые пять строк датасета
df.head()

In [ ]:
df_optimized = df.copy()

### Промежуточный вывод по данным

В результате выгрузки из базы данных получен датафрейм, содержащий 290 849 записей и 15 столбцов. Все столбцы, за исключением days_since_prev, полностью заполнены — в days_since_prev присутствует 268 909 непустых значений. Это ожидаемо, так как данный показатель рассчитывается только для повторных покупок пользователей, а для первых заказов закономерно принимает значение NULL. 

Типы данных в основном соответствуют смыслу столбцов: временные метки хранятся в формате datetime, числовые показатели — в int64 и float64, а строковые — в object. Можно преобразовать столбцы device_type_canonical и event_type_main в категориальный тип, так как они содержат ограниченное число уникальных значений, а revenue и days_since_prev перевести в float32, tickets_count — в int8

Фильтрация данных применена корректно: в выборку попали только заказы с мобильных и десктопных устройств, а мероприятия типа «Кино» исключены. Правильность расчёта days_since_prev подтверждается на конкретном примере пользователя 0005ca5e93f2cf4: между его первой покупкой (2024-07-23) и второй (2024-10-06) прошло 75 дней, что соответствует рассчитанному значению. Таким образом, выгрузка выполнена успешно, данные можно использовать для дальнейшего исследовательского анализа.

</div>

<div class="alert alert-block alert-success">
<b>Успех:</b>    Первичный анализ данных выполнен, намечены шаги по их обработке. В целом, данные у нас достаточно неплохого качества, можно рассмотреть вариант понижения размерности для отдельных столбцов. В реальной практике это иногда бывает очень полезным) 
</div>


---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


In [ ]:
#Читаем данные
rates_df = pd.read_csv('final_tickets_tenge_df.csv')

#Переименовывыем столбцы
rates_df.rename(columns={'data': 'date', 'curs': 'rate'}, inplace=True)

#Приводим даты к единому формату
df_rub = df_optimized.copy()
df_rub['order_date'] = df_rub['order_dt'].dt.date
rates_df['date'] = pd.to_datetime(rates_df['date']).dt.date

#Объединяем с курсами по дате
df_rub = df_rub.merge(
    rates_df[['date', 'rate']], 
    how='left', 
    left_on='order_date', 
    right_on='date'
)

# Пересчёт выручки в рубли
def convert_revenue(row):
    if row['currency_code'] == 'rub':
        return row['revenue']
    elif row['currency_code'] == 'kzt':
        return row['revenue'] * (row['rate'] / 100)
    else:
        return None
    
# 5. Применяем функцию и создаём новый столбец
df_rub['revenue_rub'] = df_rub.apply(convert_revenue, axis=1)

# Удаляем временные столбцы
df_rub.drop(['order_date', 'date', 'rate'], axis=1, inplace=True)
df = df_rub

In [ ]:
#Проверка
print(df_rub[['currency_code', 'revenue', 'revenue_rub']].head(10))

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

In [ ]:
#Пропущенные значения
for column in df:
    print(f'{column}: {df[column].isna().sum()}')

In [ ]:
#Пропущенные значения для каждого стол
for column in df:
    print(f'{column}: {df[column].isna().sum()/len(df)}')

In [ ]:
#Преобразуем
df['device_type_canonical'] = df['device_type_canonical'].astype('category')
df['event_type_main'] = df['event_type_main'].astype('category')
for column in ['revenue', 'days_since_prev']:
    df[column] = pd.to_numeric(df[column], downcast='float')

df['tickets_count'] = pd.to_numeric(df['tickets_count'], downcast = 'integer') 

In [ ]:
#Проверка преобразований
print(df.dtypes)

In [ ]:
#Проверка номинальных значений
for column in ['device_type_canonical', 'currency_code', 'event_type_main', 'service_name']:
    print(f'{column}:{df[column].unique()}')
    

In [ ]:
#Анализ столбца revenue_rub
df['revenue_rub'].describe()

In [ ]:
#Количество нулевых и отрицательных
df[df['revenue_rub'] <= 0].shape[0]

In [ ]:
#Их доля
df[df['revenue_rub'] <= 0].shape[0]/len(df)

In [ ]:
#Количество отрицательных
df[df['revenue_rub'] < 0].shape[0]

In [ ]:
#Пример таких данных
df[df['revenue_rub'] < 0].head(10)

In [ ]:
#Боксплот
df['revenue_rub'].plot(kind = 'box', figsize = (12,10), grid = True, vert=False)

In [ ]:
#Установка левой и правой границы
percentile_99 = df['revenue_rub'].quantile(0.99)
df = df[(df['revenue_rub'] > 0) & (df['revenue_rub'] <= percentile_99)]

In [ ]:
#Боксплот после преобразования
df['revenue_rub'].plot(kind = 'box', figsize = (12,10), grid = True, vert=False)

In [ ]:
#Анализ столбца tickets_count
df['tickets_count'].describe()

In [ ]:
df['tickets_count'].plot(kind = 'box', figsize = (12,10), grid = True, vert=False)

### Промежуточный вывод
Пропуски обнаружены только в столбце days_since_prev — 21 933 записи, остальные столбцы заполнены полностью. Проведена оптимизация типов данных: категориальные столбцы device_type_canonical и event_type_main переведены в category, числовые revenue и days_since_prev сжаты до float32, tickets_count преобразован в int8, что позволило сократить использование памяти. Проверка номинальных данных показала отсутствие скрытых пропусков и некорректных значений, все категории соответствуют ожидаемым, фильмы исключены. В столбце revenue_rub выявлено 5907 записей с нулевой или отрицательной выручкой (2,03% от общего объёма), из которых 381 являются отрицательными; после удаления нулевых и отрицательных значений, а также значений выше 99-го перцентиля, количество записей сократилось до 281 879. Выбросы в tickets_count не удалялись, так как максимальное значение 47 билетов является реалистичным для групповых заказов. Итоговый датафрейм содержит 281 879 записей и 16 столбцов, включая добавленный столбец revenue_rub.

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


In [ ]:
#Сортировка по времени совершения заказов
df = df.sort_values(by='order_ts', ascending=True)

In [ ]:
# Сводная статистика по всем пользователям
user_profile = df.groupby('user_id').agg({
    'order_dt': ['min', 'max'],
    'device_type_canonical':'first',
    'region_name':'first',
    'service_name':'first',
    'event_type_main':'first',
    'order_id':'count',
    'revenue_rub': 'mean',
    'tickets_count': 'mean',
    'days_since_prev': 'mean'
}).reset_index()

In [ ]:
user_profile.columns = [
    'user_id', 
    'first_order_dt', 
    'last_order_dt', 
    'first_device', 
    'first_region', 
    'first_service', 
    'first_event_type', 
    'orders_count', 
    'avg_revenue_rub', 
    'avg_tickets_count', 
    'avg_days_since_prev'
]

In [ ]:
user_profile['is_two'] = (user_profile['orders_count'] >= 2).astype(int)
user_profile['is_five'] = (user_profile['orders_count'] >= 5).astype(int)

In [ ]:
user_profile.head()

In [ ]:
user_profile.info()

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

In [ ]:
#Общее число пользователей
total_users = len(user_profile['user_id'])
total_users

In [ ]:
#Средняя выручка с одного заказа
user_profile['avg_revenue_rub'].mean()

In [ ]:
#Доля пользователей с 2+ заказами
share_two = user_profile['is_two'].sum() / total_users
share_two

In [ ]:
#Доля пользователей с 5+ заказами
share_five = user_profile['is_five'].sum() / total_users
share_five

In [ ]:
#Всего заказов (соотвествует данным)
user_profile['orders_count'].sum()

In [ ]:
#Анализ количества заказов
user_profile['orders_count'].describe()

In [ ]:
#Боксплот
user_profile['orders_count'].plot(kind='box', figsize=(12, 10), grid = True, vert=False)

In [ ]:
#Установка правой границы (99 квантиль)
user_profile = user_profile[(user_profile['orders_count'] <= percentile_99)]

In [ ]:
#Анализ среднего количества билетов
user_profile['avg_tickets_count'].describe()

In [ ]:
#Анализ среднего времени между заказами
user_profile['avg_days_since_prev'].describe()

### Промежуточный вывод
Общее число пользователей в выборке составило 21 700, однако после фильтрации выбросов по 99-му перцентилю их количество сократилось до 21 688. Общее количество заказов в выборке - 281 879, что полностью соответствует исходным данным после очистки от нулевой и отрицательной выручки. Средняя выручка с одного заказа равна 551,88 рубля. Доля пользователей, совершивших два и более заказа, составляет 61,67%, а доля пользователей с пятью и более заказами — 28,87%. Статистический анализ показал, что среднее количество билетов в заказе составляет 2.75, при этом максимальное значение не превышает 11 билетов, а стандартное отклонение — 0,91, что указывает на отсутствие значительных выбросов. Средний интервал между покупками равен 15,92 дня, при медианном значении 8 дней и максимальном — 148 дней. Аномалии в количестве заказов на пользователя были устранены с помощью фильтрации по 99-му перцентилю, что позволило удалить 12 экстремально активных пользователей. 

---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


In [ ]:
#по типу первого мероприятия
user_profile['first_event_type'].value_counts()

In [ ]:
#по типу устройства, с которого совершена первая покупка
user_profile['first_device'].value_counts()

In [ ]:
#по региону проведения мероприятия из первого заказа
user_profile['first_region'].value_counts().head(10)

In [ ]:
#по билетному оператору, продавшему билеты на первый заказ
user_profile['first_service'].value_counts().head(10)

### Промежуточный вывод
Распределение пользователей по всем рассмотренным сегментам является неравномерным. По типу первого мероприятия доминирующим жанром являются концерты, которые значительно опережают театры и другие жанры. По типу устройства наблюдается ярко выраженное преобладание мобильных устройств. В региональном разрезе выделяются Каменевский регион и Североярская область, которые в совокупности охватывают половину всех пользователей. Среди билетных операторов лидирует «Билеты без проблем», а топ-5 операторов обслуживают большую часть пользователей. Таким образом, все сегменты демонстрируют неравномерное распределение с чётко выраженными точками входа.

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


In [ ]:
#Доля в сегментах мероприятий
event_more = user_profile.groupby('first_event_type')['is_two'].mean().sort_values(ascending=False)
event_more

In [ ]:
#Визуал
event_more.plot(kind = 'bar', color ='orange', edgecolor='black')
plt.ylabel('Доля пользователей с 2+ заказами')
plt.xlabel('Тип мероприятия')
plt.ylim(0, 1)
plt.show()

In [ ]:
#Доля в сегментах девайсов
device_more = user_profile.groupby('first_device')['is_two'].mean().sort_values(ascending=False)
device_more

In [ ]:
#Визуал
device_more.plot(kind = 'bar', color ='orange', edgecolor='black')
plt.ylabel('Доля пользователей с 2+ заказами')
plt.xlabel('Девайс')
plt.ylim(0, 1)
plt.show()

In [ ]:
#Доля в сегментах районов
region_more = user_profile.groupby('first_region')['is_two'].mean().sort_values(ascending=False).head(10)
region_more

In [ ]:
#Визуал
region_more.plot(kind = 'barh', color ='orange', figsize = (10, 6), edgecolor='black')
plt.xlabel('Доля пользователей с 2+ заказами')
plt.ylabel('Регион')
plt.show()

In [ ]:
#Доля в сегментах сервиса
service_more = user_profile.groupby('first_service')['is_two'].mean().sort_values(ascending=False).head(10)
service_more

In [ ]:
#Визуал
service_more.plot(kind = 'barh', color ='orange', figsize = (10, 6), edgecolor='black')
plt.xlabel('Доля пользователей с 2+ заказами')
plt.ylabel('Сервис')
plt.show()

### Ответ
В ходе анализа были выделены сегменты пользователей, демонстрирующие более высокую лояльность по сравнению со средним значением по выборке (61,7%). Наиболее успешной точкой входа являются пользователи, покупающие 2–3 билета — их доля повторных заказов достигает 74%, что значительно превышает средний показатель. Среди типов мероприятий лидируют выставки (64,7%) и театр (63,9%), тогда как концерты показывают результат, близкий к среднему. В разрезе регионов наблюдается высокая вариативность: в малочисленных регионах доля возвратов может достигать 90–100%, однако основная активность сосредоточена в Каменевском регионе и Североярской области, где показатели близки к средним. Среди билетных операторов выделяются «Зе Бест!» (100% возвратов) и «Быстрый кассир» (85,2%). Таким образом, наиболее перспективными сегментами для удержания являются пользователи с 2–3 билетами, любители выставок и театра, а также клиенты операторов с высокими показателями возвращаемости.

---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

Анализ данных показывает, что тип первого мероприятия действительно влияет на вероятность повторного заказа, однако результаты не подтверждают предположение о том, что пользователи, начинающие со спорта, возвращаются чаще, чем любители концертов. Согласно полученным данным, наиболее высокая доля повторных заказов наблюдается среди пользователей, чей первый заказ был на выставки (64,7%) и театр (63,9%), тогда как у пользователей, начинавших с концертов, этот показатель чуть ниже и составляет 62,1%. Пользователи, пришедшие на спортивные мероприятия, демонстрируют наименьшую долю возвратов — всего 55,3%, что опровергает исходную гипотезу. 

Гипотеза о том, что в наиболее активных регионах доля повторных заказов выше, чем в менее активных не находит однозначного подтверждения. В регионах-лидерах по числу пользователей, таких как Каменевский регион и Североярская область, доля пользователей с повторными заказами составляет около 60–62%, что соответствует средним значениям по выборке. Однако в небольших регионах, например в Верховёрском крае и Озернопольской области, наблюдается доля возвратов близкая к 90–100%, что значительно превышает показатели крупных регионов. Это может быть связано с тем, что в малонаселённых регионах пользователи имеют меньше альтернатив для досуга и чаще возвращаются на платформу. Таким образом, высокая активность региона не является гарантией высокой лояльности пользователей, и на возвращаемость могут влиять другие факторы.


---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---


In [ ]:
#Выделяем группы
single_orders = user_profile[user_profile['is_two'] == 0]['avg_revenue_rub']
returned_orders = user_profile[user_profile['is_two'] == 1]['avg_revenue_rub']

In [ ]:
#Группа single_orders
sns.histplot(single_orders, bins=30, kde=True, color='skyblue')

plt.show()

In [ ]:
#Группа returned_orders
sns.histplot(returned_orders, bins=30, kde=True, color='salmon')

plt.show()

In [ ]:
#Визуал распределения 
plt.figure(figsize=(12, 6))
plt.hist(single_orders, bins=30, alpha=0.5, label='1 заказ', color='skyblue', density=True)
plt.hist(returned_orders, bins=30, alpha=0.5, label='2+ заказа', color='salmon', density=True)
plt.legend()
plt.title('Сравнение распределения средней выручки')
plt.xlabel('Средняя выручка (руб.)')
plt.ylabel('Плотность')
plt.show()

### Вывод по графикам 4.2.1
Анализ распределений показал, что пользователи из этих двух групп концентрируются в совершенно разных диапазонах средней выручки. У группы с одним заказом наблюдается резкий пик в самом начале шкалы, в диапазоне от нуля до трехсот рублей. Это означает, что подавляющее большинство таких пользователей совершают покупки с минимальным чеком. У пользователей, которые вернулись и сделали два и более заказа, пик распределения смещен вправо и находится в диапазоне от четырехсот до семисот рублей.

Таким образом, между группами есть четкие различия. Пользователи с повторными заказами в среднем тратят значительно больше за одну покупку, чем те, кто совершил только один заказ. Это позволяет сделать вывод, что вовлеченные и возвращающиеся клиенты приносят сервису более высокий средний чек.

---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


In [ ]:
#Формируем группы
group_2_4 = user_profile[(user_profile['orders_count'] >= 2) & (user_profile['orders_count'] <= 4)]['avg_revenue_rub']
group_5_plus = user_profile[user_profile['orders_count'] >= 5]['avg_revenue_rub']

In [ ]:
#Группа 2-4
group_2_4.describe()

In [ ]:
#Группа 5+
group_5_plus.describe()

In [ ]:
#Втзуализация распределения
bins = 30
plt.figure(figsize=(12, 6))
plt.hist(group_2_4, bins=bins, alpha=0.5, label='2–4 заказа', color='skyblue', edgecolor='black')
plt.hist(group_5_plus, bins=bins, alpha=0.5, label='5+ заказов', color='salmon', edgecolor='black')
plt.title('Сравнение распределения средней выручки')
plt.xlabel('Средняя выручка с заказа')
plt.ylabel('Количество пользователей')
plt.legend()
plt.grid(axis='y')
plt.xlim(0, 2000)  
plt.show()

### Ответ и вывод
Для каждой группы была рассчитана описательная статистика по средней выручке с заказа и построены сравнительные гистограммы распределения.

Согласно полученным данным, средняя выручка с заказа у группы с 2–4 заказами составила 557.67 рублей, в то время как у группы с 5+ заказами этот показатель немного ниже — 544.27 рублей. Медианное значение также показывает схожую картину: 477.65 рублей против 522.99 рублей. Однако стоит обратить внимание на разброс значений: у пользователей с 2–4 заказами стандартное отклонение значительно выше (420 рублей против 294 рублей), а максимальный чек достигает 2628 рублей, тогда как у второй группы максимум - 2299 рублей.

Визуальный анализ гистограмм подтверждает эти цифры. Распределение в группе 2–4 заказа (голубой цвет) является более растянутым и пологим, с большим количеством пользователей как в зоне самых низких чеков (до 200 рублей), так и в зоне высоких значений (свыше 1000 рублей). Распределение в группе 5+ заказов (розовый цвет) имеет более выраженный и узкий пик в районе 500–700 рублей, при этом в ней значительно меньше пользователей с самыми низкими и самыми высокими средними чеками.

Различия между этими двумя группами есть, но они касаются не столько среднего значения, сколько характера поведения. Пользователи с 5+ заказами более стабильны и предсказуемы — их средняя выручка находится в более узком диапазоне. Пользователи с 2–4 заказами более разнородны: среди них чаще встречаются как те, кто покупает дешевые товары, так и те, кто делает действительно крупные покупки.

---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

In [ ]:
#Анализ avg_tickets_count
print("Статистика по среднему количеству билетов:")
print(user_profile['avg_tickets_count'].describe())

In [ ]:
#Гистограмма распределения
plt.figure(figsize=(10, 5))
plt.hist(user_profile['avg_tickets_count'], bins=30, color='skyblue', edgecolor='black')
plt.title('Распределение пользователей по среднему количеству билетов в заказе')
plt.xlabel('Среднее количество билетов')
plt.ylabel('Количество пользователей')
plt.grid(axis='y')
plt.show()

In [ ]:
#Создаём сегменты

def segment_tickets (count):
    if count < 2:
        return '1–2 билета'
    elif count < 3:
        return '2–3 билета'
    elif count < 5:
        return '3–5 билетов'
    else:
        return '5+ билетов'

user_profile['tickets_segment'] = user_profile['avg_tickets_count'].apply(segment_tickets)

In [ ]:
#Расчёт метрик по сегментам
tickets_segment_stats = user_profile.groupby('tickets_segment').agg(
    users_count=('user_id', 'count'),
    share_returned=('is_two', 'mean')
).sort_values('users_count', ascending=False)

In [ ]:
#Доли
tickets_segment_stats['share_of_total'] = tickets_segment_stats['users_count'] / len(user_profile)

In [ ]:
#Результат разделения
display(tickets_segment_stats)

In [ ]:
# Количество пользователей визуал
tickets_segment_stats['users_count'].plot(kind='bar', color='darkred', edgecolor='black')
plt.title('Количество пользователей по сегментам')
plt.ylabel('Количество')
plt.grid(axis='y')
plt.show()

In [ ]:
#Доля вернувшихся визуал
tickets_segment_stats['share_returned'].plot(kind='bar', color='orange', edgecolor='black')
plt.title('Доля пользователей с 2+ заказами по сегментам')
plt.ylabel('Доля')
plt.ylim(0, 1)
plt.grid(axis='y')
plt.show()

### Ответы
Распределение пользователей по сегментам крайне неравномерно: 85% сосредоточены в группах с 2–3 и 3–5 билетами, а сегменты 1–2 и 5+ билетов малочисленны. При этом доля повторных покупок аномально высока в сегменте 2–3 билета (74%) и резко снижается до 54% в группе 3–5 билетов, достигая минимума среди покупателей 5+ билетов (19%). Это указывает на то, что наиболее лояльная аудитория — пользователи с небольшими компаниями, а крупные групповые покупки совершаются редко и не способствуют возврату.

---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


In [ ]:
#Преобразуем дату первого заказа в день недели
user_profile['first_order_weekday'] = pd.to_datetime(user_profile['first_order_dt']).dt.day_name()

In [ ]:
#Группировка по дню недели
weekday_stats = user_profile.groupby('first_order_weekday').agg(
    users_count=('user_id', 'count'),
    share_returned=('is_two', 'mean')
).reset_index()

In [ ]:
display(weekday_stats)

In [ ]:
#Визуал по первой покупке
weekday_stats[['users_count']].plot(kind='bar', color='green', edgecolor='black', legend=False)
plt.title('Количество пользователей по дню первой покупки')
plt.xlabel('День недели')
plt.ylabel('Количество')
plt.grid(axis='y')
plt.show()

In [ ]:
#Визуал по вернувшимся
weekday_stats[['share_returned']].plot(kind='bar', color='orange', edgecolor='black', legend=False)
plt.title('Доля вернувшихся по дню первой покупки')
plt.xlabel('День недели')
plt.ylabel('Доля')
plt.ylim(0, 1)
plt.grid(axis='y')
plt.show()

#### Ответ
День недели первой покупки практически не влияет на возврат: доля вернувшихся колеблется от 59,5% до 64,1% (разница всего 4,6 п.п.). Наибольшая лояльность у тех, кто пришёл в субботу (64,1%), наименьшая — в четверг (59,6%). Распределение новых пользователей по дням равномерное, значимых различий нет.

---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


In [ ]:
#Создание групп по кол-ву заказов
def order_group(count):
    if count >= 2 and count <= 4:
        return '2–4 заказа'
    elif count >= 5:
        return '5+ заказов'
    else:
        return '1 заказ'

user_profile['order_group'] = user_profile['orders_count'].apply(order_group)

#Проверка
print(user_profile['order_group'].value_counts())

In [ ]:
#Средний интервал между заказами для групп
interval_stats = user_profile[user_profile['order_group'].isin(['2–4 заказа', '5+ заказов'])].groupby('order_group').agg(
    users_count=('user_id', 'count'),
    avg_interval=('avg_days_since_prev', 'mean'),
    median_interval=('avg_days_since_prev', 'median'),
    std_interval=('avg_days_since_prev', 'std')
).reset_index()

display(interval_stats)

In [ ]:
interval_stats[['avg_interval', 'median_interval']].plot(kind='bar', figsize=(10, 6), color=['skyblue', 'orange'], edgecolor='black')
plt.title('Средний и медианный интервал между заказами')
plt.ylabel('Дней')
plt.xlabel('Группа пользователей')
plt.grid(axis='y')
plt.show()

### Вывод
Чем короче интервал между заказами, тем выше вероятность повторного возврата: у пользователей с 5+ заказами средний интервал составляет 9,6 дня, а у группы 2–4 заказа — 21,2 дня, что более чем в два раза больше. Медианные значения также ниже у более активной группы (7,8 против 9 дней), а стандартное отклонение у них значительно меньше (7,8 против 28,3). 

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
#Данные
corr_data = user_profile[['orders_count', 'first_event_type', 'first_device', 'first_region', 'first_service', 'avg_revenue_rub', 'avg_tickets_count']].copy()

In [ ]:
#Перевод категорий в строковые
for col in ['first_event_type', 'first_device', 'first_region', 'first_service']:
    corr_data[col] = corr_data[col].astype(str)

In [ ]:
corr_matrix = corr_data.phik_matrix(interval_cols=['orders_count', 'avg_revenue_rub', 'avg_tickets_count'])
corr_matrix

In [ ]:
#Визуал
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.show()

In [ ]:
#Сегменты
def order_segment(count):
    if count == 1:
        return '1 заказ'
    elif count <= 4:
        return '2–4 заказа'
    else:
        return '5+ заказов'

user_profile['order_segment'] = user_profile['orders_count'].apply(order_segment)


In [ ]:
#Проверка
print(user_profile['order_segment'].value_counts())

In [ ]:
#Данныe
corr_data = user_profile[['orders_count', 'first_event_type', 'first_device', 
                          'first_region', 'first_service', 'avg_revenue_rub', 
                          'avg_tickets_count', 'order_segment']].copy()

In [ ]:
#Визуал 2-4 заказа
seg = corr_data[corr_data['order_segment'] == '2–4 заказа'].drop('order_segment', axis=1)
sns.heatmap(seg.phik_matrix(interval_cols=['orders_count', 'avg_revenue_rub', 'avg_tickets_count']), 
            annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Сегмент 2–4 заказа')
plt.show()

In [ ]:
#5+ визуал
seg = corr_data[corr_data['order_segment'] == '5+ заказов'].drop('order_segment', axis=1)
sns.heatmap(seg.phik_matrix(interval_cols=['orders_count', 'avg_revenue_rub', 'avg_tickets_count']), 
            annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Сегмент 5+ заказов')
plt.show()

#### Вывод
Анализ корреляционной матрицы показывает, что наиболее значимым признаком, связанным с количеством заказов, является регион первого мероприятия (first_region), особенно в сегменте пользователей с пятью и более заказами, где коэффициент корреляции достигает 0,49. Среднее количество билетов (avg_tickets_count) показывает умеренную связь (0,37) только в сегменте 2–4 заказа, но теряет значимость для более активных пользователей. Средняя выручка (avg_revenue_rub) имеет слабую связь с числом заказов в обоих сегментах (0,11). Тип первого мероприятия, устройство и билетный оператор практически не влияют на количество заказов, так как их коэффициенты близки к нулю. 

Таким образом, ключевым фактором, влияющим на долгосрочную активность пользователей, является регион.

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

## Общий вывод и рекомендации

В ходе исследования были проанализированы данные о покупках пользователей сервиса «Яндекс Афиша» за 2024 год. Итоговая выборка после очистки составила 21 688 пользователей и 281 879 заказов. Данные были подготовлены: выручка в тенге переведена в рубли по курсу на дату заказа, удалены заказы с нулевой и отрицательной выручкой, а также проведена фильтрация выбросов по 99-му перцентилю. В результате средняя выручка с одного заказа составила 551,9 рубля, среднее количество билетов в заказе — 2,75, а средний интервал между покупками — 15,9 дня.

Среди признаков, влияющих на возврат пользователей, наиболее значимым оказался регион первого мероприятия: пользователи из Каменевского региона и Североярской области составляют почти половину аудитории и демонстрируют более высокую лояльность. Количество билетов также связано с повторными покупками: пользователи, приобретающие 2–3 билета, возвращаются чаще (74%), чем покупатели 1–2 билетов (51%). При этом средняя выручка слабо влияет на вероятность повторного заказа.

Временной анализ показал, что критический период для удержания клиента составляет до двух недель: пользователи с 5+ заказами возвращаются в среднем через 9,6 дня, тогда как с 2–4 заказами — через 21,2 дня. День недели первой покупки практически не влияет на возврат.

Корреляционный анализ подтвердил, что регион первого мероприятия имеет наибольшую связь с числом заказов (коэффициент 0,49 в сегменте 5+ заказов). Тип мероприятия, устройство и билетный оператор практически не связаны с частотой покупок.

Рекомендации заказчику:

1. Сосредоточить усилия на удержании пользователей в первые две недели после первого заказа.

2. Усилить маркетинговую активность в Каменевском регионе и Североярской области, где сосредоточена наиболее активная аудитория, а также развивать мобильный канал как основной (83% пользователей приходят с мобильных устройств).

3. Стимулировать пользователей с 1–2 билетами переходить в сегмент 2–3 билета через акции на парные билеты и персональные подборки мероприятий для компаний.
